In [ ]:
%pip install pandas scikit-learn plotly nbformat

In [ ]:
from pathlib import Path
import pandas as pd

# Localiza o pickle exportado pela limpeza.
caminho_dados = Path('_DadosLimpos/jogos_steam.pkl.gz')
if not caminho_dados.is_file():
    caminho_dados = Path('..') / caminho_dados

# As colunas de tokens ja sao listas; nao precisam de json.loads.
df = pd.read_pickle(caminho_dados)
df.head()


In [ ]:
# Bag of Words: contagem dos termos nas descrições sem stopwords.
from sklearn.feature_extraction.text import CountVectorizer

descricoes_sem_stopwords = df['description_tokens_sem_stopwords']

# Usa os tokens existentes, sem tokenizar o texto novamente.
vectorizer_bow = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow = vectorizer_bow.fit_transform(descricoes_sem_stopwords)

# Mantém a matriz esparsa para economizar memória; linhas seguem a ordem de df.
print(f'Bag of Words: {X_bow.shape[0]} jogos e {X_bow.shape[1]} termos')

df_bow = pd.DataFrame.sparse.from_spmatrix(
    X_bow,
    index=pd.MultiIndex.from_frame(df[['appid', 'name']]),
    columns=vectorizer_bow.get_feature_names_out(),
)
# Exibe 5 jogos e até 20 termos presentes nesses jogos.
colunas_preview = (X_bow[:5].getnnz(axis=0) > 0).nonzero()[0][:20]
df_bow.iloc[:5, colunas_preview]

In [ ]:
# TF-IDF: ponderação dos termos nas mesmas descrições sem stopwords.
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df['description_tokens_sem_stopwords']
)

tfidf_palavras = tfidf_vectorizer.get_feature_names_out()

# Mantem os dados esparsos para evitar a alocacao de 157 GiB.
df_tfidf = pd.DataFrame.sparse.from_spmatrix(
    tfidf_matrix,
    columns=tfidf_palavras
).fillna(0.0)

df_tfidf.insert(0, 'appid', df['appid'].values)

# Mostra apenas termos presentes nos primeiros 20 jogos.
colunas_preview_tfidf = (tfidf_matrix[:20].getnnz(axis=0) > 0).nonzero()[0][:20]
preview_tfidf = df_tfidf.iloc[:20, [0] + (colunas_preview_tfidf + 1).tolist()]
from IPython.display import display
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(preview_tfidf)

In [ ]:
# Similaridade do cosseno - Bag of Words: um jogo contra todo o catalogo.
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Altere o appid para consultar outro jogo (10 = Counter-Strike).
appid_referencia_bow = 10
top_n_bow = 10
posicoes_bow = (df['appid'] == appid_referencia_bow).to_numpy().nonzero()[0]
if len(posicoes_bow) == 0:
    raise ValueError('O appid de referencia nao foi encontrado no DataFrame.')
posicao_bow = int(posicoes_bow[0])
if X_bow[posicao_bow].nnz == 0:
    raise ValueError('O jogo de referencia nao possui termos para comparar.')

# Gera apenas N valores, evitando uma matriz N x N de todos os pares.
similaridade_bow = cosine_similarity(
    X_bow[posicao_bow], X_bow
).ravel()

df_similaridade_bow = df[['appid', 'name']].copy()
df_similaridade_bow['similaridade_cosseno'] = similaridade_bow
# Exclui o proprio jogo do ranking, preservando a ordem original no DataFrame completo.
recomendacoes_bow = (
    df_similaridade_bow
    .iloc[[i for i in range(len(df)) if i != posicao_bow]]
    .nlargest(top_n_bow, 'similaridade_cosseno')
)
print('Referencia:', df.iloc[posicao_bow]['name'], '| Bag of Words')
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(recomendacoes_bow)


In [ ]:
# Similaridade do cosseno - TF-IDF: um jogo contra todo o catalogo.
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Altere o appid para consultar outro jogo (10 = Counter-Strike).
appid_referencia_tfidf = 10
top_n_tfidf = 10
posicoes_tfidf = (df['appid'] == appid_referencia_tfidf).to_numpy().nonzero()[0]
if len(posicoes_tfidf) == 0:
    raise ValueError('O appid de referencia nao foi encontrado no DataFrame.')
posicao_tfidf = int(posicoes_tfidf[0])
if tfidf_matrix[posicao_tfidf].nnz == 0:
    raise ValueError('O jogo de referencia nao possui termos para comparar.')

# Gera apenas N valores, evitando uma matriz N x N de todos os pares.
similaridade_tfidf = cosine_similarity(
    tfidf_matrix[posicao_tfidf], tfidf_matrix
).ravel()

df_similaridade_tfidf = df[['appid', 'name']].copy()
df_similaridade_tfidf['similaridade_cosseno'] = similaridade_tfidf
# Exclui o proprio jogo do ranking, preservando a ordem original no DataFrame completo.
recomendacoes_tfidf = (
    df_similaridade_tfidf
    .iloc[[i for i in range(len(df)) if i != posicao_tfidf]]
    .nlargest(top_n_tfidf, 'similaridade_cosseno')
)
print('Referencia:', df.iloc[posicao_tfidf]['name'], '| TF-IDF')
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(recomendacoes_tfidf)


In [ ]:
# Popularidade = total de avaliacoes positivas + negativas (SteamSpy/Steam).
# Fontes: https://steamspy.com/api.php e https://partner.steamgames.com/doc/store/getreviews
# Ranking restrito aos jogos do catalogo com contagem disponivel no cache.
import json
from pathlib import Path
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

caminho_popularidade = caminho_dados.parent / 'popularidade_steamspy.json'
if not caminho_popularidade.is_file():
    raise FileNotFoundError('Execute 3_Analise_Clusters/coletar_popularidade.py para gerar o cache.')
cache_popularidade = json.loads(caminho_popularidade.read_text(encoding='utf-8'))
popularidade = pd.DataFrame(cache_popularidade['jogos'].values())
if 'fonte' not in popularidade:
    popularidade['fonte'] = cache_popularidade['fonte']
else:
    popularidade['fonte'] = popularidade['fonte'].fillna(cache_popularidade['fonte'])
if popularidade.empty:
    raise ValueError('O cache nao possui contagens de avaliacoes.')

# Posicao explicita para manter o alinhamento com as matrizes vetorizadas.
base_generos = df[['appid', 'name', 'genres']].copy()
base_generos['posicao_matriz'] = range(len(df))
base_generos['genero'] = base_generos['genres'].fillna('').str.split(';')
base_generos = base_generos.explode('genero')
base_generos['genero'] = base_generos['genero'].str.strip()
base_generos = base_generos[base_generos['genero'].ne('')].drop_duplicates(['genero', 'appid'])
base_generos = base_generos.merge(popularidade, on='appid', how='left', validate='many_to_one')

# Nao preenche contagens desconhecidas com zero.
cobertura_generos = base_generos.groupby('genero').agg(
    jogos_no_catalogo=('appid', 'size'), jogos_com_contagem=('total_avaliacoes', 'count')
)
cobertura_generos['sem_contagem'] = cobertura_generos['jogos_no_catalogo'] - cobertura_generos['jogos_com_contagem']
print('Cobertura da fonte externa (contagens ausentes ficam fora do ranking):')
display(cobertura_generos)

# Considera cada genero do jogo; desempata pelo appid.
top_10_por_genero = (
    base_generos.dropna(subset=['total_avaliacoes'])
    .sort_values(['genero', 'total_avaliacoes', 'appid'], ascending=[True, False, True])
    .groupby('genero', sort=False).head(10).copy()
)
top_10_por_genero['total_avaliacoes'] = top_10_por_genero['total_avaliacoes'].astype('int64')
top_10_por_genero['posicao_no_genero'] = top_10_por_genero.groupby('genero').cumcount() + 1
print('Ate 10 jogos por genero, conforme a cobertura da fonte:')
display(top_10_por_genero[['genero', 'posicao_no_genero', 'appid', 'name', 'total_avaliacoes', 'coletado_em_utc', 'fonte']])

# Cada jogo aparece uma unica vez nos mapas, mesmo se integrar varios rankings.
jogos_amostra = top_10_por_genero.drop_duplicates('appid').copy()
if jogos_amostra.empty:
    raise ValueError('Nenhum jogo com contagem disponivel para a visualizacao.')
posicoes_amostra = jogos_amostra['posicao_matriz'].to_numpy(dtype=int)
quantidade_jogos = len(jogos_amostra)
rotulos_amostra = [f'{nome} ({appid})' for appid, nome in jogos_amostra[['appid', 'name']].itertuples(index=False, name=None)]
print(f'{quantidade_jogos} jogos unicos selecionados; mesma ordem nos dois mapas.')

# Mesma escala (0 a 1) nos dois graficos para comparar as intensidades.
def plotar_similaridade(matriz, titulo):
    fig = go.Figure(go.Heatmap(
        z=matriz.to_numpy(),
        x=matriz.columns.tolist(),
        y=matriz.index.tolist(),
        colorscale='Blues',
        zmin=0,
        zmax=1,
        texttemplate='%{z:.2f}' if len(matriz) <= 20 else '',
        textfont=dict(size=10),
        colorbar=dict(title='Similaridade'),
        hovertemplate=(
            'Jogo da linha: %{y}<br>'
            'Jogo da coluna: %{x}<br>'
            'Similaridade: %{z:.4f}<extra></extra>'
        ),
    ))
    fig.update_layout(
        title=titulo,
        template='plotly_white',
        width=1200,
        height=1100,
        margin=dict(l=280, r=100, t=90, b=300),
        xaxis=dict(tickangle=-60, tickfont=dict(size=10), automargin=True),
        yaxis=dict(autorange='reversed', tickfont=dict(size=10), automargin=True),
    )
    fig.show()


In [ ]:
# Calcula somente os pares dos jogos selecionados por popularidade.
# Reutiliza a vetorizacao do catalogo completo, sem recalcular o vocabulario ou IDF.
similaridade_amostra_bow = cosine_similarity(X_bow[posicoes_amostra])
df_similaridade_amostra_bow = pd.DataFrame(
    similaridade_amostra_bow, index=rotulos_amostra, columns=rotulos_amostra
)
plotar_similaridade(
    df_similaridade_amostra_bow,
    f'Bag of Words - similaridade entre {quantidade_jogos} jogos dos rankings por genero (avaliacoes SteamSpy/Steam)'
)


In [ ]:
# Calcula somente os pares dos jogos selecionados por popularidade.
# Reutiliza a vetorizacao do catalogo completo, sem recalcular o vocabulario ou IDF.
similaridade_amostra_tfidf = cosine_similarity(tfidf_matrix[posicoes_amostra])
df_similaridade_amostra_tfidf = pd.DataFrame(
    similaridade_amostra_tfidf, index=rotulos_amostra, columns=rotulos_amostra
)
plotar_similaridade(
    df_similaridade_amostra_tfidf,
    f'TF-IDF - similaridade entre {quantidade_jogos} jogos dos rankings por genero (avaliacoes SteamSpy/Steam)'
)


# Comparando Clusters e categorias

In [ ]:
from sklearn.cluster import KMeans

# Quantidade inicial de clusters para explorar as descricoes dos jogos.
k = 10
print(f'K = {k} clusters')

# Dez inicializacoes e semente fixa para resultados reproduziveis.
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(tfidf_matrix)

df['cluster'] = clusters

df[
    ['appid', 'name', 'genres', 'cluster']
].sort_values('cluster')

In [ ]:
# Um jogo pode contribuir para varios generos, mas apenas um cluster.
generos_clusters = df[['appid', 'genres', 'cluster']].copy()
generos_clusters['genero'] = generos_clusters['genres'].fillna('').str.split(';')
generos_clusters = generos_clusters.explode('genero')
generos_clusters['genero'] = generos_clusters['genero'].str.strip()
generos_clusters = generos_clusters[generos_clusters['genero'].ne('')]
generos_clusters = generos_clusters.drop_duplicates(['appid', 'genero', 'cluster']).reset_index(drop=True)
comparacao_clusters = pd.crosstab(
    generos_clusters['genero'], generos_clusters['cluster']
)
comparacao_clusters


# Visualizando os clusters com PCA


In [ ]:
# PCA sobre matriz esparsa, sem materializar todo o catalogo em memoria.
from sklearn.decomposition import PCA
import plotly.express as px
from html import escape
from textwrap import wrap


def formatar_texto_pca(texto, largura=65):
    if pd.isna(texto) or not str(texto).strip():
        return 'Nao informado'
    # Quebra em linhas sem cortar a descricao; escapa caracteres HTML do texto.
    linhas = wrap(' '.join(str(texto).split()), width=largura)
    return '<br>'.join(escape(linha) for linha in linhas)


def melhorar_hover_pca(figura):
    figura.update_traces(hovertemplate=(
        '<b>%{customdata[0]}</b><br>'
        'AppID: %{customdata[1]} | Cluster: %{customdata[2]}<br>'
        '<b>Generos:</b> %{customdata[3]}<br><br>'
        '<b>Descricao</b><br>%{customdata[4]}'
        '<extra></extra>'
    ))
    figura.update_layout(hoverlabel=dict(
        bgcolor='white', bordercolor='#64748b',
        font=dict(size=14, family='Arial', color='#0f172a'),
        align='left', namelength=-1,
    ))


pca_2d = PCA(n_components=2, svd_solver='arpack', random_state=42)

coordenadas_2d = pca_2d.fit_transform(tfidf_matrix)

df_pca_2d = pd.DataFrame({
    "appid": df['appid'],
    "name": df['name'],
    "genres": df['genres'],
    "description": df['description'],
    "cluster": df['cluster'].astype(str),
    "PCA1": coordenadas_2d[:, 0],
    "PCA2": coordenadas_2d[:, 1],
})

# Formata apenas os textos de exibicao, preservando as descricoes originais.
df_pca_2d['nome_hover'] = df_pca_2d['name'].map(formatar_texto_pca)
df_pca_2d['generos_hover'] = df_pca_2d['genres'].map(formatar_texto_pca)
df_pca_2d['descricao_hover'] = df_pca_2d['description'].map(formatar_texto_pca)

fig = px.scatter(
    df_pca_2d,
    x="PCA1",
    y="PCA2",
    color="cluster",
    custom_data=['nome_hover', 'appid', 'cluster', 'generos_hover', 'descricao_hover'],
    title="Clusters K-Means em PCA (2D)",
)

fig.update_traces(marker=dict(size=12))

fig.update_layout(
    xaxis_title="Componente Principal 1",
    yaxis_title="Componente Principal 2",
    legend_title="Cluster",
)

melhorar_hover_pca(fig)
fig.show()

## Visualização em 3 Dimensões

In [ ]:
# PCA sobre matriz esparsa, sem materializar todo o catalogo em memoria.
# Reduzir o TF-IDF para três dimensões
pca_3d = PCA(n_components=3, svd_solver='arpack', random_state=42)

coordenadas_3d = pca_3d.fit_transform(
    tfidf_matrix
)

# Criar DataFrame para visualização
df_pca_3d = pd.DataFrame({
    "appid": df['appid'],
    "name": df['name'],
    "genres": df['genres'],
    "description": df['description'],
    "cluster": df['cluster'].astype(str),
    "PCA1": coordenadas_3d[:, 0],
    "PCA2": coordenadas_3d[:, 1],
    "PCA3": coordenadas_3d[:, 2],
})

# Criar gráfico 3D
# Formata apenas os textos de exibicao, preservando as descricoes originais.
df_pca_3d['nome_hover'] = df_pca_3d['name'].map(formatar_texto_pca)
df_pca_3d['generos_hover'] = df_pca_3d['genres'].map(formatar_texto_pca)
df_pca_3d['descricao_hover'] = df_pca_3d['description'].map(formatar_texto_pca)

fig = px.scatter_3d(
    df_pca_3d,
    x="PCA1",
    y="PCA2",
    z="PCA3",
    color="cluster",
    custom_data=['nome_hover', 'appid', 'cluster', 'generos_hover', 'descricao_hover'],
    title="Clusters das descrições — PCA em 3 dimensões"
)

fig.update_traces(
    marker=dict(size=6)
)

fig.update_layout(
    scene=dict(
        xaxis_title="Componente principal 1",
        yaxis_title="Componente principal 2",
        zaxis_title="Componente principal 3"
    ),
    legend_title="Cluster"
)

melhorar_hover_pca(fig)
fig.show()

In [ ]:
print(
    "Variância explicada em 2D:",
    round(pca_2d.explained_variance_ratio_.sum(), 3)
)

print(
    "Variância explicada em 3D:",
    round(pca_3d.explained_variance_ratio_.sum(), 3)
)

# Quais palavras representam cada cluster?

In [ ]:
import numpy as np

# Nomes das palavras do vocabulário TF-IDF
termos = tfidf_vectorizer.get_feature_names_out()

# Centróides calculados pelo K-Means
centroides = kmeans.cluster_centers_

# Quantidade de termos que queremos mostrar
n_termos = 8

resultados_clusters = []

for cluster_id in range(k):

    # Ordenar os termos pelo peso no centróide
    indices_ordenados = np.argsort(
        centroides[cluster_id]
    )[::-1]

    principais_indices = indices_ordenados[:n_termos]

    principais_termos = termos[principais_indices]
    principais_pesos = centroides[
        cluster_id,
        principais_indices
    ]

    resultados_clusters.append({
        "cluster": cluster_id,
        "tokens_representativos": ", ".join(principais_termos)
    })

# Criar DataFrame
df_tokens_clusters = pd.DataFrame(resultados_clusters)

df_tokens_clusters

In [ ]:
# Quantidade e exemplos de jogos por cluster; evita listar milhares de nomes.
limite_exemplos = 10
jogos_por_cluster = (
    df.groupby('cluster')
      .agg(
          n_jogos=('appid', 'size'),
          exemplos_jogos=('name', lambda nomes: ', '.join(nomes.head(limite_exemplos).fillna('(sem nome)'))),
      )
      .reset_index()
)
df_interpretacao_clusters = df_tokens_clusters.merge(
    jogos_por_cluster, on='cluster', how='left', validate='one_to_one'
)
df_interpretacao_clusters


# Coesão interna dos Clusters

In [ ]:
import numpy as np
from sklearn.preprocessing import normalize

# Coesao = media do cosseno entre pares distintos de jogos no cluster.
# Para vetores normalizados: soma dos pares = ||soma dos vetores||^2
# menos a soma das normas ao quadrado. Evita uma matriz N x N.
# Vetores nulos contribuem com similaridade zero, como em cosine_similarity.
if tfidf_matrix.shape[0] != len(df):
    raise ValueError('TF-IDF e df devem ter as mesmas linhas, na mesma ordem.')
resultados_coesao = []
for cluster_id in sorted(df['cluster'].dropna().unique()):
    posicoes = np.flatnonzero(df['cluster'].eq(cluster_id).to_numpy())
    vetores = normalize(tfidf_matrix[posicoes], norm='l2', copy=True)
    n_jogos = len(posicoes)
    coesao_media = np.nan  # Nao existem pares em clusters com um unico jogo.
    if n_jogos > 1:
        soma_vetores = np.asarray(vetores.sum(axis=0)).ravel()
        soma_diagonal = float(vetores.multiply(vetores).sum())
        coesao_media = float(np.clip(
            (soma_vetores @ soma_vetores - soma_diagonal)
            / (n_jogos * (n_jogos - 1)), 0.0, 1.0
        ))
    resultados_coesao.append({
        'cluster': cluster_id,
        'n_jogos': n_jogos,
        'coesao_media': coesao_media,
    })
df_coesao = pd.DataFrame(resultados_coesao)
df_coesao


In [ ]:
# Resumo da coesao com exemplos dos jogos de cada grupo.
df_coesao_jogos = df_coesao.merge(
    jogos_por_cluster[['cluster', 'exemplos_jogos']],
    on='cluster', validate='one_to_one'
)
df_coesao_jogos.sort_values('coesao_media', ascending=False)


# Distância entre os Clusters

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Similaridade entre os centróides
similaridade_centroides = cosine_similarity(
    kmeans.cluster_centers_
)

# Converter similaridade em distância
distancia_centroides = 1 - similaridade_centroides

# Criar DataFrame
df_distancia_clusters = pd.DataFrame(
    distancia_centroides,
    index=[f"Cluster {i}" for i in range(k)],
    columns=[f"Cluster {i}" for i in range(k)]
)

df_distancia_clusters.round(3)

In [ ]:
resultados_distancias = []

for i in range(k):
    for j in range(i + 1, k):

        resultados_distancias.append({
            "cluster_1": i,
            "cluster_2": j,
            "distancia": distancia_centroides[i, j]
        })

df_pares_clusters = pd.DataFrame(
    resultados_distancias
)

df_pares_clusters.sort_values("distancia")

## Clusters mais próximos e mais distantes

In [ ]:
# Mostra ate limite_exemplos jogos de cada cluster comparado.
# Identificar menor e maior distância entre clusters
par_mais_proximo = df_pares_clusters.loc[
    df_pares_clusters["distancia"].idxmin()
]

par_mais_distante = df_pares_clusters.loc[
    df_pares_clusters["distancia"].idxmax()
]


def jogos_do_cluster(cluster_id):
    return df.loc[
        df["cluster"] == cluster_id,
        "name"
    ].head(limite_exemplos).fillna("(sem nome)").tolist()


# Clusters mais próximos
c1_prox = int(par_mais_proximo["cluster_1"])
c2_prox = int(par_mais_proximo["cluster_2"])

# Clusters mais distantes
c1_dist = int(par_mais_distante["cluster_1"])
c2_dist = int(par_mais_distante["cluster_2"])


print("CLUSTERS MAIS PRÓXIMOS")
print("----------------------")
print(
    f"Cluster {c1_prox} × Cluster {c2_prox}"
)
print(
    f"Distância: {par_mais_proximo['distancia']:.3f}"
)
print(
    f"Cluster {c1_prox}: {', '.join(jogos_do_cluster(c1_prox))}"
)
print(
    f"Cluster {c2_prox}: {', '.join(jogos_do_cluster(c2_prox))}"
)


print("\nCLUSTERS MAIS DISTANTES")
print("-----------------------")
print(
    f"Cluster {c1_dist} × Cluster {c2_dist}"
)
print(
    f"Distância: {par_mais_distante['distancia']:.3f}"
)
print(
    f"Cluster {c1_dist}: {', '.join(jogos_do_cluster(c1_dist))}"
)
print(
    f"Cluster {c2_dist}: {', '.join(jogos_do_cluster(c2_dist))}"
)